Behind pipeline function
- 3 stages
- tokenzier -> model -> postprocesing

Tokenizer 
- split text into tokens
- add speical topens like cls, sep
- convert them to unique input ids (js id, no meaning captured)


In [4]:
from transformers import AutoTokenizer, AutoModel

checkpoint = 'distilbert-base-uncased-finetuned-sst-2-english'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

raw_inputs = ["I hate you but i still love you"]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)
outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

# output a high dim tensor that represents the input passed
# automodel loads the body of model without the head
# padding adds [pad] token to shorten setences to match longest sentnece length
# attention mask -> treat peadding tokens as masked 0
# truncation true to truncate excess tokens to max allowed length in block size
# togethere tokenzier takes raw input text cover to rows of same length token ids
# they are public classes in transformer liubrary

c:\Users\Admin\anaconda3\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


{'input_ids': tensor([[ 101, 1045, 5223, 2017, 2021, 1045, 2145, 2293, 2017,  102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
torch.Size([1, 10, 768])


In [11]:
from transformers import AutoModelForSequenceClassification
import torch
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)
print(outputs.logits.shape)
print(outputs.logits)
# raw unnormalised scores, need go throuigh softmax to become prob
pred = torch.nn.functional.softmax(outputs.logits,dim=-1)
print(pred)
model.config.id2label

torch.Size([1, 2])
tensor([[-3.8892,  4.1132]], grad_fn=<AddmmBackward0>)
tensor([[3.3453e-04, 9.9967e-01]], grad_fn=<SoftmaxBackward0>)


{0: 'NEGATIVE', 1: 'POSITIVE'}

instantiate a transformer model
specfit a config file and model file
first lopen config file to read the config class
config -> blueprint for the model
save pretrained model using save pretrained method and load it using from pretrained

Architecture vs weights vs model vs checkpoint (cake analogy)
- architecture = the recipe (structure only: how many layers, hidden size, etc) - no learning happened yet
- weights = the specific measured ingredients from ONE baking attempt (the actual learned numbers from a training run)
- model = recipe + measured ingredients TOGETHER = an actual cake you can eat (architecture + weights, ready to run)
- checkpoint = a saved snapshot - the recipe card with a specific batch's amounts written down and saved to a file,
  so anyone can reload it later and recreate the EXACT same cake without re-baking (re-training) from scratch

note: same architecture can have many different checkpoints
- eg distilbert-base-uncased (generic) vs distilbert-base-uncased-finetuned-sst-2-english (same shape, further trained/fine-tuned weights for sentiment)

mapping to code
- config.json = architecture (blueprint)
- model.safetensors = weights (learned params)
- checkpoint string (eg "distilbert-base-uncased-finetuned-sst-2-english") = name of a saved config+weights pair on the HF Hub
- from_pretrained(checkpoint):
    1. reads config -> builds the architecture (empty shell)
    2. loads the saved weights into that shell
    3. result = a ready-to-use model, no training needed

In [ ]:
# from pretrained will download and cache data from hf or ur own model filke
# checkpoint -> specifc model arch and weights
model.save_pretrained("/Downloads")
# save model onto a directory
# will create 2 files, config.,json and safe tensoirs
# config.json will have all attributes to build model (blueprint)
# safe tensors contain all weights
# these 2 tgt build the overall model, one for archtetirue, one for params

Encoding text
- deep dive to tokenzing
- passing in raw textr to tokenzier 
- returens input id, token tytpe id, atten mask
- token tytpe id -> whiich part of input is a, which is b
- mask -> which should be attended to

handling multiple seq
- models always expect a BATCH dimension, even for 1 sentence
  (single sentence still needs to be wrapped: [ids], shape (1, T), not just ids, shape (T,))
- batching = stacking multiple sequences into ONE tensor call, each sequence = one row
- eg batch of 2 sentences -> tensor shape (2, T) -> sent through model in a single forward pass
- each row one seq
- together matrix is one batch
